# MTSE 6850 — Midterm Assignment
## Reproduce & Extend: Ward et al. (2016)

**Name:** ___________________________  
**UFID:** ___________________________  
**Due:** End of Week 9 (Sunday 11:59 PM) via Canvas

---

### Overview

This midterm asks you to:

1. **Part A — Setup & Data** (not graded separately): Reproduce the dataset used in Ward et al. 2016
2. **Part B — Reproduce** (40 pts): Re-implement the bandgap regression pipeline from the paper and evaluate it honestly
3. **Part C — Extend** (40 pts): Implement **one** of three provided extensions
4. **Part D — Write-up** (20 pts): Interpret your results in a short scientific write-up

**Total: 100 pts = 20% of your course grade**

---

### Paper reference

> Ward, L., Agrawal, A., Choudhary, A., & Wolverton, C. (2016).  
> **A general-purpose machine learning framework for predicting properties of inorganic materials.**  
> *npj Computational Materials*, 2, 16028.  
> https://doi.org/10.1038/npjcompumats.2016.28

Download the paper from the Canvas assignment page before starting.

---

### Submission instructions

- Run **Kernel → Restart & Run All** before submitting. All cells must have output.
- Submit this `.ipynb` file to Canvas. **Do not submit a PDF.**
- Part D write-up goes in the markdown cells provided — do not submit a separate document.
- If you use your extension option's provided starter code, you must add at least 5 substantive cells beyond it.

---

### Academic integrity

All code and written analysis must be your own. You may use course materials, the paper, and public documentation (scikit-learn, pymatgen, matminer). Discussing approaches with classmates is fine; sharing code or written analysis is not. If you use AI tools to help debug, note this at the top of the relevant cell.

---
## Part A — Setup & Data

This section is **not separately graded** — it is scaffolding to get you the dataset you need for Parts B and C. Run all cells and confirm the final dataset loads before proceeding.

**What Ward et al. did:** They collected bandgap values from the Materials Project for a large set of inorganic compounds, featurized them using composition-based descriptors (the Magpie descriptor set), and trained a Random Forest regressor. Their key result was an MAE of ~0.45 eV on a random 80/20 split.

**What you will do:** Reproduce this using the current Materials Project API and the Matminer Magpie featurizer. Your exact numbers will differ slightly from the paper due to database updates since 2016 — that is expected and not a problem.

In [ ]:
# Cell A1 — Imports
# Run this cell first. If any import fails, check your conda environment.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
import os

# scikit-learn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Materials Project
from mp_api.client import MPRester

# Matminer featurizers
from matminer.featurizers.composition import ElementProperty
from matminer.featurizers.base import MultipleFeaturizer
from pymatgen.core import Composition

# Load API key
load_dotenv()
API_KEY = os.getenv('MP_API_KEY')

if API_KEY:
    print(f'✅ All imports successful')
    print(f'   API key loaded ({len(API_KEY)} characters)')
else:
    print('❌ API key not found — check your .env file')

In [ ]:
# Cell A2 — Fetch bandgap data from Materials Project
# This fetches inorganic compounds with computed bandgaps.
# Expected runtime: 30–90 seconds depending on your connection.
# Ward et al. used ~3,900 compounds. We will use a comparable set.

print('Fetching data from Materials Project...')
print('(This may take 30–90 seconds)')

with MPRester(API_KEY) as mpr:
    docs = mpr.summary.search(
        num_elements=(2, 4),          # binary to quaternary compounds
        band_gap=(0.01, 10.0),        # semiconductors and insulators only
        is_stable=True,               # thermodynamically stable entries
        fields=[
            'material_id',
            'formula_pretty',
            'band_gap',
            'formation_energy_per_atom',
            'energy_above_hull',
            'nsites',
            'volume',
            'symmetry',
        ]
    )

print(f'✅ Retrieved {len(docs)} entries')

In [ ]:
# Cell A3 — Build DataFrame and add Composition objects

rows = []
for d in docs:
    rows.append({
        'material_id':   d.material_id,
        'formula':       d.formula_pretty,
        'band_gap':      d.band_gap,
        'formation_energy_per_atom': d.formation_energy_per_atom,
        'energy_above_hull': d.energy_above_hull,
        'nsites':        d.nsites,
        'spacegroup_number': d.symmetry.number if d.symmetry else None,
        'crystal_system':   str(d.symmetry.crystal_system) if d.symmetry else None,
    })

df = pd.DataFrame(rows)

# Parse formula into Pymatgen Composition objects (required for Matminer)
df['composition'] = df['formula'].apply(Composition)

print(f'Dataset shape: {df.shape}')
print(f'Band gap range: {df.band_gap.min():.3f} – {df.band_gap.max():.3f} eV')
print(f'Mean band gap:  {df.band_gap.mean():.3f} eV')
print()
df.head()

In [ ]:
# Cell A4 — Featurize with Magpie descriptors (Ward et al. descriptor set)
# The Magpie descriptor set uses statistics of elemental properties
# (mean, range, min, max, mode, variance) computed over all elements
# present in the composition. This is a composition-only descriptor:
# it does not use any structural information.

print('Generating Magpie composition descriptors...')
print('(This may take 1–3 minutes for large datasets)')

# Ward et al. used the 'magpie' preset from ElementProperty
featurizer = ElementProperty.from_preset('magpie')
featurizer.featurize_dataframe(df, col_id='composition', ignore_errors=True, inplace=True)

# Identify feature columns (all Magpie descriptor columns)
feature_cols = featurizer.feature_labels()

# Drop rows with any NaN features
df_clean = df.dropna(subset=feature_cols + ['band_gap']).reset_index(drop=True)

print(f'\n✅ Featurization complete')
print(f'   Entries before cleaning: {len(df)}')
print(f'   Entries after cleaning:  {len(df_clean)}')
print(f'   Number of Magpie features: {len(feature_cols)}')
print(f'\nFirst 5 feature names: {feature_cols[:5]}')

In [ ]:
# Cell A5 — Save featurized dataset to CSV (avoids re-fetching if notebook restarts)

CACHE_FILE = 'midterm_dataset.csv'

# Save columns that can be serialized to CSV (drop Composition objects)
cols_to_save = ['material_id', 'formula', 'band_gap', 'formation_energy_per_atom',
                'energy_above_hull', 'nsites', 'spacegroup_number', 'crystal_system'] + feature_cols
df_clean[cols_to_save].to_csv(CACHE_FILE, index=False)

print(f'✅ Dataset saved to {CACHE_FILE}')
print(f'   Shape: {df_clean[cols_to_save].shape}')
print()
print('If you restart this notebook, you can reload from the cache:')
print("  df_clean = pd.read_csv('midterm_dataset.csv')")
print("  feature_cols = [c for c in df_clean.columns if 'MagpieData' in c or 'Egap' in c]")
print('  # Or re-run this cell after A4 to regenerate')

In [ ]:
# Cell A6 — Quick EDA on the dataset

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Bandgap distribution
axes[0].hist(df_clean['band_gap'], bins=60, color='#2E75B6', edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Band Gap (eV)', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Bandgap distribution', fontsize=12)
axes[0].axvline(df_clean['band_gap'].mean(), color='#E24B4A', linestyle='--',
                label=f'Mean = {df_clean["band_gap"].mean():.2f} eV')
axes[0].legend(fontsize=9)

# Crystal system composition
crystal_counts = df_clean['crystal_system'].value_counts()
axes[1].barh(crystal_counts.index, crystal_counts.values, color='#1F3864')
axes[1].set_xlabel('Count', fontsize=11)
axes[1].set_title('Crystal system breakdown', fontsize=12)

# Bandgap vs. formation energy
sc = axes[2].scatter(
    df_clean['formation_energy_per_atom'],
    df_clean['band_gap'],
    alpha=0.4, s=8, c=df_clean['nsites'],
    cmap='viridis'
)
plt.colorbar(sc, ax=axes[2], label='# sites')
axes[2].set_xlabel('Formation energy (eV/atom)', fontsize=11)
axes[2].set_ylabel('Band gap (eV)', fontsize=11)
axes[2].set_title('Bandgap vs. formation energy', fontsize=12)

plt.suptitle(f'Dataset overview: {len(df_clean)} compounds', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('midterm_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plot saved as midterm_eda.png')

---
## Part B — Reproduce (40 pts)

Reproduce the core result from Ward et al. 2016: predict bandgap from Magpie composition descriptors using a Random Forest regressor, and evaluate on a random 80/20 train-test split.

**Rubric:**
| Criterion | Points |
|---|---|
| B1: Correct train/test split (random, stratified by crystal system) | 10 |
| B2: Random Forest trained and evaluated, MAE and R² reported | 10 |
| B3: Parity plot (predicted vs. actual) with correct formatting | 10 |
| B4: Feature importance plot, top 15 features labeled | 10 |

**Target:** Your MAE should fall in the range 0.35–0.65 eV. If it falls outside this range, add a markdown cell explaining why (data version differences, dataset size, etc.).

In [ ]:
# Cell B1 — Train/test split
# Ward et al. used a random 80/20 split.
# Use random_state=42 for reproducibility.

X = df_clean[feature_cols].values
y = df_clean['band_gap'].values

# TODO: Create an 80/20 train/test split using train_test_split.
# Use random_state=42.

# YOUR CODE HERE
X_train, X_test, y_train, y_test = None, None, None, None

# ── Validation checks (do not modify) ────────────────────────────────────────
assert X_train is not None, 'X_train is None — complete the split above'
print(f'Training set:   {X_train.shape[0]} samples, {X_train.shape[1]} features')
print(f'Test set:       {X_test.shape[0]} samples')
print(f'Train fraction: {X_train.shape[0] / (X_train.shape[0] + X_test.shape[0]):.2f}')

In [ ]:
# Cell B2 — Train Random Forest and evaluate
# Ward et al. used n_estimators=500. Match this as closely as possible.
# Note: 500 trees may take 1–3 minutes. Start with n_estimators=100 to
# verify your code works, then increase to 500 for final submission.

# TODO: Instantiate and train a RandomForestRegressor.
# Parameters to match Ward et al.:
#   n_estimators=500, random_state=42
# All other parameters at sklearn defaults.

# YOUR CODE HERE
rf = None  # replace with your model

# Fit and predict (do not modify this block)
assert rf is not None, 'rf is None — instantiate your model above'
print('Training Random Forest...')
rf.fit(X_train, y_train)
y_pred_test  = rf.predict(X_test)
y_pred_train = rf.predict(X_train)

# Compute metrics
mae_test  = mean_absolute_error(y_test,  y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_test   = r2_score(y_test,  y_pred_test)
mae_train = mean_absolute_error(y_train, y_pred_train)

print(f'\n=== Test set performance ===')
print(f'  MAE:  {mae_test:.4f} eV')
print(f'  RMSE: {rmse_test:.4f} eV')
print(f'  R²:   {r2_test:.4f}')
print(f'\n=== Train set performance (for overfitting check) ===')
print(f'  MAE:  {mae_train:.4f} eV')
print(f'\n=== Comparison to Ward et al. target ===')
print(f'  Ward et al. reported MAE ≈ 0.45 eV (random split)')
print(f'  Your MAE: {mae_test:.4f} eV  (delta = {abs(mae_test - 0.45):.4f} eV)')

In [ ]:
# Cell B3 — Parity plot (predicted vs. actual bandgap)
# A parity plot is the standard way to visualize regression model performance
# in materials science papers. It must include:
#   - A diagonal 1:1 line (perfect prediction)
#   - ±0.5 eV error bands (shaded)
#   - MAE and R² annotated on the plot
#   - Colored by crystal system (use df_clean['crystal_system'] for test indices)

# Get crystal system labels for test set
# Note: you need to track which rows went into the test set.
# Use the indices from the split, or re-split using df_clean directly.

# TODO: Create the parity plot described above.
# Required elements (each missing element loses 2-3 points):
#   1. Scatter: y_test on x-axis, y_pred_test on y-axis
#   2. 1:1 diagonal line
#   3. ±0.5 eV error bands (shaded)
#   4. MAE and R² annotated in plot text
#   5. Axis labels with units
#   6. Title

# YOUR CODE HERE

plt.tight_layout()
plt.savefig('midterm_parity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Parity plot saved as midterm_parity.png')

In [ ]:
# Cell B4 — Feature importance plot
# Random Forest provides feature_importances_ attribute (mean decrease in impurity).
# Plot the top 15 most important features as a horizontal bar chart.

# TODO: Create a horizontal bar chart of the top 15 feature importances.
# Requirements:
#   1. Use rf.feature_importances_ paired with feature_cols labels
#   2. Sort descending, show only top 15
#   3. Bars colored by descriptor family (hint: all Magpie features contain
#      the property name — e.g. 'MagpieData mean Electronegativity')
#   4. X-axis label: 'Mean decrease in impurity'
#   5. Title: 'Top 15 feature importances — Magpie RF model'

# YOUR CODE HERE

plt.tight_layout()
plt.savefig('midterm_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature importance plot saved as midterm_importance.png')

### B5 — Brief comparison to Ward et al. (write here)

*In 3–5 sentences: How does your MAE compare to the paper's reported value? What might explain any differences (data version, dataset size, hyperparameters, split strategy)? Is the R² consistent with a useful predictive model?*

**Your answer:**

> [Write your response here. Replace this text.]

---
## Part C — Extend (40 pts)

Choose **exactly one** of the three extensions below. Implement it fully. Each extension is worth 40 pts and requires:
1. Working code that reproduces or compares results
2. At least one new visualization
3. A written interpretation in the provided markdown cell

**Set your chosen extension here:**

In [ ]:
# Cell C0 — Declare your chosen extension
# Set EXTENSION to 1, 2, or 3

EXTENSION = None  # ← change to 1, 2, or 3

assert EXTENSION in [1, 2, 3], 'Set EXTENSION to 1, 2, or 3 in this cell'
print(f'Extension {EXTENSION} selected.')

ext_names = {
    1: 'XGBoost + hyperparameter tuning',
    2: 'Prototype-aware train/test splitting',
    3: 'Structural feature enrichment'
}
print(f'You are implementing: {ext_names[EXTENSION]}')

---
### Extension 1 — XGBoost + Hyperparameter Tuning

**Goal:** Replace the Random Forest with XGBoost and tune at least 3 hyperparameters using cross-validation. Report whether tuning improves the test MAE relative to your Part B baseline.

**Rubric (40 pts):**
| Task | Points |
|---|---|
| XGBoost model trained, test MAE reported | 10 |
| At least 3 hyperparameters tuned via CV (GridSearchCV or RandomizedSearchCV) | 10 |
| Learning curve plot (train and validation MAE vs. n_estimators) | 10 |
| Written interpretation comparing RF vs. XGBoost performance | 10 |

*Skip this section if you are doing Extension 2 or 3.*

In [ ]:
# Cell C1a — Extension 1: XGBoost baseline
# (Only complete if EXTENSION == 1)

if EXTENSION == 1:
    # Install XGBoost if needed: conda install -c conda-forge xgboost
    import xgboost as xgb
    print(f'XGBoost version: {xgb.__version__}')

    # TODO: Train a baseline XGBoost model with default parameters
    # using the same X_train, X_test, y_train, y_test from Part B.
    # Report MAE on the test set.

    # YOUR CODE HERE
    pass
else:
    print(f'Skipping Extension 1 (you selected Extension {EXTENSION})')

In [ ]:
# Cell C1b — Extension 1: Hyperparameter tuning
# Tune at least 3 of: n_estimators, max_depth, learning_rate,
# subsample, colsample_bytree, reg_alpha, reg_lambda
# Use 5-fold CV and report best parameters and CV MAE.

if EXTENSION == 1:
    # TODO: Use GridSearchCV or RandomizedSearchCV to tune XGBoost.
    # Report: best parameters, CV MAE, test MAE with best parameters.

    # YOUR CODE HERE
    pass
else:
    print(f'Skipping Extension 1 (you selected Extension {EXTENSION})')

In [ ]:
# Cell C1c — Extension 1: Learning curve
# Plot train MAE and CV validation MAE as a function of n_estimators.
# This shows where the model starts to overfit (if it does).

if EXTENSION == 1:
    # Hint: XGBoost supports early stopping with eval_set.
    # Or: loop over a range of n_estimators values and record train/val MAE.

    # TODO: Plot train and validation MAE vs. n_estimators.
    # Label the optimal n_estimators value on the plot.

    # YOUR CODE HERE
    plt.tight_layout()
    plt.savefig('midterm_ext1_learning_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'Skipping Extension 1 (you selected Extension {EXTENSION})')

### Extension 1 — Written interpretation

*Answer both questions:*
1. *Does XGBoost outperform Random Forest on this task? By how much? Why might one perform better than the other for this specific dataset?*
2. *What did the learning curve reveal about overfitting? What is the optimal n_estimators for your tuned model?*

**Your answer:**

> [Write your response here. Replace this text. Minimum 150 words.]

---
### Extension 2 — Prototype-Aware Train/Test Splitting

**Motivation:** Ward et al. used a random train/test split. A known problem with random splits in materials ML is that structurally similar compounds (same crystal prototype, e.g. all rocksalt compounds) end up in both train and test sets. This inflates accuracy because the model essentially memorizes the pattern for that structure type.

**Goal:** Re-evaluate the Ward et al. model using a prototype-aware split — train on some crystal prototypes, test on others. Compare the MAE between random and prototype-aware splits.

**Rubric (40 pts):**
| Task | Points |
|---|---|
| Prototype-aware split implemented correctly (by spacegroup number or crystal system) | 12 |
| MAE compared: random split vs. prototype-aware split, with statistical significance | 12 |
| Visualization showing MAE by crystal system (random vs. prototype-aware) | 8 |
| Written interpretation of the leakage effect | 8 |

*Skip this section if you are doing Extension 1 or 3.*

In [ ]:
# Cell C2a — Extension 2: Prototype-aware split
# Strategy: split by crystal system (7 systems: cubic, hexagonal,
# tetragonal, orthorhombic, trigonal, monoclinic, triclinic).
# Hold out 2 crystal systems for testing; train on the remaining 5.
# This ensures no crystal system overlap between train and test.

if EXTENSION == 2:
    # Check available crystal systems in your dataset
    print('Crystal systems in dataset:')
    print(df_clean['crystal_system'].value_counts())
    print()

    # TODO: Implement a prototype-aware split.
    # Step 1: Select 2 crystal systems to hold out as the test set.
    # Step 2: Create X_train_proto, X_test_proto, y_train_proto, y_test_proto
    #         based on crystal system membership.
    # Step 3: Train the same RandomForest (n_estimators=500, random_state=42)
    #         on X_train_proto and evaluate on X_test_proto.

    # YOUR CODE HERE
    pass
else:
    print(f'Skipping Extension 2 (you selected Extension {EXTENSION})')

In [ ]:
# Cell C2b — Extension 2: Compare random vs. prototype-aware MAE
# Run the random split multiple times (n_repeats=10, different random seeds)
# to get a distribution of MAE values. Compare to the prototype-aware MAE.

if EXTENSION == 2:
    # TODO:
    # 1. Run 10 repeated random splits, record test MAE for each.
    # 2. Report mean ± std of random-split MAE.
    # 3. Report prototype-aware split MAE.
    # 4. Conclude whether the difference is scientifically meaningful.

    # YOUR CODE HERE
    pass
else:
    print(f'Skipping Extension 2 (you selected Extension {EXTENSION})')

In [ ]:
# Cell C2c — Extension 2: MAE by crystal system
# For the random split model, compute MAE separately for each crystal system
# in the test set. Which crystal systems are predicted well vs. poorly?

if EXTENSION == 2:
    # TODO: Create a grouped bar chart showing:
    # - X-axis: crystal system
    # - Y-axis: test MAE (eV)
    # - Two bars per group: random-split model vs. prototype-aware model
    #   (for prototype-aware model, only the held-out test systems have bars)
    # Annotate with the number of test compounds per system.

    # YOUR CODE HERE
    plt.tight_layout()
    plt.savefig('midterm_ext2_by_crystal.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'Skipping Extension 2 (you selected Extension {EXTENSION})')

### Extension 2 — Written interpretation

*Answer both questions:*
1. *How much does the MAE increase when switching from random to prototype-aware splitting? What does this tell you about the generalizability of the Ward et al. model to unseen crystal structure types?*
2. *Which crystal systems are hardest to predict (highest MAE)? Can you offer a physical or chemical explanation for why those systems might be harder?*

**Your answer:**

> [Write your response here. Replace this text. Minimum 150 words.]

---
### Extension 3 — Structural Feature Enrichment

**Motivation:** Ward et al. used only composition-based (Magpie) descriptors. Crystal structure information — bond lengths, coordination environments, local geometry — is physically more informative for bandgap prediction. Can adding structural features improve the model?

**Goal:** Augment the Magpie descriptor set with structure-based descriptors from Matminer. Train a new model on the enriched feature set and compare to the composition-only baseline.

**Important caveat:** Structure-based featurization requires relaxed structures from Materials Project, which takes longer. You will work with a smaller subset (~500 compounds) for this extension.

**Rubric (40 pts):**
| Task | Points |
|---|---|
| Structure-based features computed for a ≥300 compound subset | 10 |
| New model trained on Magpie + structure features, MAE reported | 10 |
| Feature importance comparison: which structural features matter most? | 10 |
| Written interpretation: does structure information help and why? | 10 |

*Skip this section if you are doing Extension 1 or 2.*

In [ ]:
# Cell C3a — Extension 3: Fetch structures for a subset of compounds
# We work with a subset because structure featurization is slow.
# Take a random sample of 500 compounds from your dataset.

if EXTENSION == 3:
    # Sample 500 compounds for structure featurization
    df_subset = df_clean.sample(n=min(500, len(df_clean)), random_state=42).reset_index(drop=True)
    print(f'Working with a subset of {len(df_subset)} compounds')
    print(f'Fetching crystal structures from Materials Project...')
    print('(This may take 2–5 minutes)')

    # Fetch structures for the subset
    material_ids = df_subset['material_id'].tolist()
    structures = {}

    with MPRester(API_KEY) as mpr:
        # Fetch in batches of 50 to avoid timeout
        batch_size = 50
        for i in range(0, len(material_ids), batch_size):
            batch = material_ids[i:i+batch_size]
            docs_batch = mpr.summary.search(
                material_ids=batch,
                fields=['material_id', 'structure']
            )
            for doc in docs_batch:
                structures[doc.material_id] = doc.structure
            if (i // batch_size + 1) % 2 == 0:
                print(f'  Fetched {min(i+batch_size, len(material_ids))}/{len(material_ids)}')

    df_subset['structure'] = df_subset['material_id'].map(structures)
    df_subset = df_subset.dropna(subset=['structure']).reset_index(drop=True)
    print(f'\n✅ Structures fetched for {len(df_subset)} compounds')
else:
    print(f'Skipping Extension 3 (you selected Extension {EXTENSION})')

In [ ]:
# Cell C3b — Extension 3: Compute structure-based features
# Suggested featurizers (pick at least 2):
#   - SiteStatsFingerprint (local bonding environments)
#   - RadialDistributionFunction (bond length statistics)
#   - StructuralHeterogeneity (variation in local environments)
#   - MaximumPackingEfficiency
# All are available in matminer.featurizers.structure

if EXTENSION == 3:
    from matminer.featurizers.structure import (
        SiteStatsFingerprint,
        StructuralHeterogeneity,
        MaximumPackingEfficiency
    )
    from matminer.featurizers.site import CrystalNNFingerprint

    # TODO:
    # 1. Apply at least 2 structure-based featurizers to df_subset.
    # 2. Identify the new structure feature column names.
    # 3. Drop any rows with NaN values.
    # 4. Print the number of structure features added.

    # Hint: featurizer.featurize_dataframe(df_subset, col_id='structure',
    #         ignore_errors=True, inplace=True)

    # YOUR CODE HERE
    structure_feature_cols = []  # populate with your structure feature column names
    pass
else:
    print(f'Skipping Extension 3 (you selected Extension {EXTENSION})')

In [ ]:
# Cell C3c — Extension 3: Train and compare models
# Train two models on the subset:
#   Model A: Magpie features only (composition)
#   Model B: Magpie + structure features
# Use the same RF parameters (n_estimators=500) and 80/20 split.

if EXTENSION == 3:
    # TODO:
    # 1. Split df_subset into train/test (80/20, random_state=42)
    # 2. Train Model A (Magpie only) on the subset
    # 3. Train Model B (Magpie + structure) on the subset
    # 4. Report and compare test MAE for both models
    # 5. Create a bar chart comparing the two MAEs

    # YOUR CODE HERE
    plt.tight_layout()
    plt.savefig('midterm_ext3_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'Skipping Extension 3 (you selected Extension {EXTENSION})')

In [ ]:
# Cell C3d — Extension 3: Feature importance for enriched model
# Which structural features matter most?
# Create a feature importance plot for Model B, highlighting
# structure-based features (e.g. with different color) vs. Magpie features.

if EXTENSION == 3:
    # TODO: Feature importance bar chart for Model B.
    # Show top 20 features, color structure features differently from
    # composition features so it's easy to see which descriptor type
    # dominates the model.

    # YOUR CODE HERE
    plt.tight_layout()
    plt.savefig('midterm_ext3_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'Skipping Extension 3 (you selected Extension {EXTENSION})')

### Extension 3 — Written interpretation

*Answer both questions:*
1. *Does adding structural features improve the MAE? If yes, by how much and which features contribute most? If no, why might composition alone be sufficient for this task?*
2. *The Magpie descriptor set is composition-only — it ignores where atoms are actually positioned. For which types of materials (think about the structure-property relationship) would you expect structural descriptors to matter most?*

**Your answer:**

> [Write your response here. Replace this text. Minimum 150 words.]

---
## Part D — Write-up (20 pts)

Write your interpretation of the full midterm in the cells below. This is a scientific write-up, not a code summary. Minimum word count is indicated per question.

**Rubric:**
| Question | Points |
|---|---|
| D1 — Scientific significance of the result | 5 |
| D2 — Critical assessment of the Ward et al. methodology | 7 |
| D3 — What you would do differently | 8 |

### D1 — Scientific significance (~100 words minimum)

*Why does bandgap prediction matter scientifically and technologically? Who benefits from having a fast, accurate bandgap predictor that doesn't require DFT calculations?*

**Your answer:**

> [Write your response here. Replace this text.]

### D2 — Critical assessment of Ward et al. methodology (~200 words minimum)

*Identify two methodological strengths and two methodological limitations in the Ward et al. 2016 approach. Be specific — reference specific choices in the paper (descriptor set, split strategy, evaluation metric, dataset construction). Do not just say "the model is good" or "more data would help."*

**Your answer:**

> [Write your response here. Replace this text.]

### D3 — What you would do differently (~200 words minimum)

*Based on what you learned in this midterm and in the course so far: if you were continuing this project for your own research, what is the single most important improvement you would make to the Ward et al. pipeline? Why? How would you evaluate whether your improvement actually worked?*

*Hint: Your Extension choice is a natural starting point, but push beyond it. Think about the physical problem, the data, and the validation strategy.*

**Your answer:**

> [Write your response here. Replace this text.]

---
## Submission Checklist

Before submitting, confirm all of the following:

- [ ] Cell C0 has `EXTENSION` set to 1, 2, or 3
- [ ] All cells in Part A have been run and have output
- [ ] Part B cells B1–B4 are complete with correct output
- [ ] Part B5 markdown reflection is filled in
- [ ] Part C extension cells are complete with output
- [ ] Part C interpretation markdown is filled in (≥150 words)
- [ ] Part D write-up D1, D2, D3 are all filled in
- [ ] **Kernel → Restart & Run All** executed — all cells have fresh output
- [ ] No cell raises an error (warnings are OK)
- [ ] Notebook submitted to Canvas as `.ipynb` (not PDF)